# Graph State Encrypted Cloning Verification

## Can Arbitrary Maximally Entangled Graph States Serve as Resources?

This notebook validates that the encrypted cloning protocol extends beyond 1D linear cluster states to **arbitrary graph states**, provided there exists a bipartition into Signal ($S$) and Noise ($N_{\text{noise}}$) qubits such that the state exhibits maximal entanglement across the cut.

### Hypotheses Tested

| # | Hypothesis | Status |
|---|-----------|--------|
| A | A **2k-qubit** graph state (with full-rank biadjacency) can clone an arbitrary **k-qubit** state | To verify |
| B | A **2n-qubit** graph state can produce **n encrypted clones** of a single qubit state | To verify |

### Tested Graphs
- $N=4$: Cycle Graph $C_4$, Random Connected Bipartite
- $N=6$: Generalized Network (Perfect matching + extra connections)

In [2]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np
from numpy import kron, eye, sqrt, pi
from functools import reduce
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=120)

def mk(*args):
    return reduce(kron, args)

def partial_trace(rho, dims, keep):
    n = len(dims)
    rho_r = rho.reshape(dims + dims)
    trace_axes = sorted(set(range(n)) - set(keep))
    for i, ax in enumerate(sorted(trace_axes, reverse=True)):
        rho_r = np.trace(rho_r, axis1=ax, axis2=ax + n - i)
    d_keep = [dims[k] for k in sorted(keep)]
    return rho_r.reshape(int(np.prod(d_keep)), int(np.prod(d_keep)))

def fidelity(rho, sigma_m):
    sq_rho = np.linalg.eigh(rho)
    vals = np.maximum(sq_rho[0], 0)
    vecs = sq_rho[1]
    sq = vecs @ np.diag(np.sqrt(vals)) @ vecs.conj().T
    product = sq @ sigma_m @ sq
    vals2 = np.maximum(np.linalg.eigvalsh(product), 0)
    return (np.sum(np.sqrt(vals2)))**2

def von_neumann_entropy(rho):
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-15]
    return -np.sum(eigenvalues * np.log2(eigenvalues))

def random_pure_state(dim=2):
    psi = np.random.randn(dim) + 1j * np.random.randn(dim)
    return psi / np.linalg.norm(psi)

def random_density_matrix(dim=2):
    psi = random_pure_state(dim)
    return np.outer(psi, psi.conj())

def build_permutation_matrix(n_qubits, perm):
    dim = 2**n_qubits
    P = np.zeros((dim, dim))
    for idx in range(dim):
        bits = [(idx >> (n_qubits-1-q)) & 1 for q in range(n_qubits)]
        new_bits = [bits[perm[q]] for q in range(n_qubits)]
        new_idx = sum(b << (n_qubits-1-q) for q, b in enumerate(new_bits))
        P[new_idx, idx] = 1.0
    return P

I2 = eye(2, dtype=complex)
sigma = [
    I2,
    np.array([[0,1],[1,0]], dtype=complex),
    np.array([[0,-1j],[1j,0]], dtype=complex),
    np.array([[1,0],[0,-1]], dtype=complex),
]
bell_phi_plus = np.array([1, 0, 0, 1], dtype=complex) / sqrt(2)

print("All imports and helper functions loaded ✓")

All imports and helper functions loaded ✓


## 1. Generalized Graph State Construction

In [3]:
def build_graph_state(N, edges):
    '''Build an N-qubit graph state given a list of edges.'''
    plus = np.array([1, 1], dtype=complex) / sqrt(2)
    state = plus.copy()
    for i in range(1, N):
        state = np.kron(state, plus)
    for u, v in edges:
        for idx in range(2**N):
            b_u = (idx >> (N-1-u)) & 1
            b_v = (idx >> (N-1-v)) & 1
            if b_u == 1 and b_v == 1:
                state[idx] *= -1
    return state

test_graphs = {
    4: [
        {
            'name': 'Cycle Graph C_4',
            'edges': [(0,1), (1,2), (2,3), (3,0)],
            'signal_qubits': [0, 1],
            'noise_qubits': [2, 3]
        },
        {
            'name': 'Random Max-Entangled Bipartite',
            'edges': [(0, 2), (0, 3), (1, 2)], 
            'signal_qubits': [0, 1],
            'noise_qubits': [2, 3]
        }
    ],
    6: [
        {
            'name': 'Generalized Network',
            'edges': [(0,3), (1,4), (2,5), (0,4), (1,5)],
            'signal_qubits': [0, 1, 2],
            'noise_qubits': [3, 4, 5]
        }
    ]
}

print("=" * 70)
print("Graph State Construction and Entanglement Verification")
print("=" * 70)

for N, graphs in test_graphs.items():
    k = N // 2
    for graph in graphs:
        psi = build_graph_state(N, graph['edges'])
        perm = graph['signal_qubits'] + graph['noise_qubits']
        P = build_permutation_matrix(N, perm)
        psi_reordered = P @ psi

        rho_reordered = np.outer(psi_reordered, psi_reordered.conj())
        rho_S = partial_trace(rho_reordered, [2**k, 2**k], [0])

        state_mat = psi_reordered.reshape(2**k, 2**k)
        _, sv, _ = np.linalg.svd(state_mat)
        schmidt_rank = int(np.sum(sv > 1e-10))
        is_max_ent = np.allclose(rho_S, eye(2**k) / 2**k)

        print(f"\n  [{graph['name']}] N={N}")
        print(f"    Edges: {graph['edges']}")
        print(f"    Signal: {graph['signal_qubits']}, Noise: {graph['noise_qubits']}")
        print(f"    Schmidt rank = {schmidt_rank} (need {2**k})")
        print(f"    rho_S = I/{2**k}: {'✅' if is_max_ent else '❌'}")

Graph State Construction and Entanglement Verification

  [Cycle Graph C_4] N=4
    Edges: [(0, 1), (1, 2), (2, 3), (3, 0)]
    Signal: [0, 1], Noise: [2, 3]
    Schmidt rank = 4 (need 4)
    rho_S = I/4: ✅

  [Random Max-Entangled Bipartite] N=4
    Edges: [(0, 2), (0, 3), (1, 2)]
    Signal: [0, 1], Noise: [2, 3]
    Schmidt rank = 4 (need 4)
    rho_S = I/4: ✅

  [Generalized Network] N=6
    Edges: [(0, 3), (1, 4), (2, 5), (0, 4), (1, 5)]
    Signal: [0, 1, 2], Noise: [3, 4, 5]
    Schmidt rank = 8 (need 8)
    rho_S = I/8: ✅


## 2. Encoding and Decoding Operators

In [4]:
def build_single_qubit_codec(n_clones=1):
    alpha = [1, 1j, -(1j)**(n_clones+1), 1j]
    n = n_clones
    dim = 2**(n+1)

    U_enc = np.zeros((dim, dim), dtype=complex)
    for mu in range(4):
        pauli_prod = reduce(kron, [sigma[mu]] * (n+1))
        U_enc += (1/alpha[mu]) * pauli_prod
    U_enc /= 2

    phi = bell_phi_plus
    U_dec = np.zeros((dim, dim), dtype=complex)
    for mu in range(4):
        phi_mu = kron(sigma[mu], I2) @ phi
        proj_mu = np.outer(phi_mu, phi_mu.conj())
        if n >= 2:
            sigma_T_prod = reduce(kron, [sigma[mu].T] * (n-1))
        else:
            sigma_T_prod = np.array([[1.]], dtype=complex)
        U_dec += alpha[mu] * kron(proj_mu, sigma_T_prod)
    return U_enc, U_dec

def extract_UN(resource_state_reordered, k):
    d = 2**k
    state_mat = resource_state_reordered.reshape(d, d)
    v = state_mat * d
    L, S_vals, Rh = np.linalg.svd(v.conj())
    U_N = L @ Rh
    return U_N, S_vals

U_enc1, U_dec1 = build_single_qubit_codec(n_clones=1)

def build_k_qubit_bell_decoder(k):
    U_dec_raw = U_dec1.copy()
    for j in range(1, k):
        U_dec_raw = kron(U_dec_raw, U_dec1)
    target_order = list(range(0, 2*k, 2)) + list(range(1, 2*k, 2))
    perm = [0] * (2*k)
    for new_pos, old_pos in enumerate(target_order):
        perm[new_pos] = old_pos
    P = build_permutation_matrix(2*k, perm)
    return P @ U_dec_raw @ P.conj().T

def build_k_qubit_encoder(k):
    U_enc_raw = U_enc1.copy()
    for j in range(1, k):
        U_enc_raw = kron(U_enc_raw, U_enc1)
    target_order = list(range(0, 2*k, 2)) + list(range(1, 2*k, 2))
    perm = [0] * (2*k)
    for new_pos, old_pos in enumerate(target_order):
        perm[new_pos] = old_pos
    P = build_permutation_matrix(2*k, perm)
    return P @ U_enc_raw @ P.conj().T

def build_graph_decoder(N, graph):
    k = N // 2
    psi = build_graph_state(N, graph['edges'])
    perm = graph['signal_qubits'] + graph['noise_qubits']
    P = build_permutation_matrix(N, perm)
    psi_reordered = P @ psi

    U_N, sv = extract_UN(psi_reordered, k)
    U_dec_bell = build_k_qubit_bell_decoder(k)
    d = 2**k
    transform = kron(eye(d), U_N)
    full_decoder = U_dec_bell @ transform
    return full_decoder, U_N, U_dec_bell, sv

## 3. Protocol Verification

In [5]:
def run_k_qubit_protocol(k, rho_in, graph_decoder, U_enc_k, graph):
    N = 2 * k
    total_q = 3 * k
    dims = [2] * total_q
    d_source = 2**k

    psi_graph = build_graph_state(N, graph['edges'])
    perm = graph['signal_qubits'] + graph['noise_qubits']
    P_graph = build_permutation_matrix(N, perm)
    psi_reordered = P_graph @ psi_graph
    rho_resource = np.outer(psi_reordered, psi_reordered.conj())

    state = kron(rho_in, rho_resource)
    U_enc_full = kron(U_enc_k, eye(d_source))
    state_enc = U_enc_full @ state @ U_enc_full.conj().T

    U_dec_full = kron(eye(d_source), graph_decoder)
    state_dec = U_dec_full @ state_enc @ U_dec_full.conj().T

    keep_signal = list(range(k, 2*k))
    rho_recovered = partial_trace(state_dec, dims, keep=keep_signal)

    rho_S_enc = partial_trace(state_enc, dims, keep=keep_signal)
    enc_ok = np.allclose(rho_S_enc, eye(d_source)/d_source, atol=1e-8)

    return rho_recovered, enc_ok, state_enc, state_dec

print("=" * 70)
print("HYPOTHESIS A: 2k-qubit graph → k-qubit cloning (1 clone)")
print("=" * 70)

np.random.seed(42)

for N, graphs in test_graphs.items():
    k = N // 2
    for graph in graphs:
        print(f"\n{'─'*60}")
        print(f"  Testing Graph: {graph['name']} (k={k})")
        print(f"{'─'*60}")

        U_enc_k = build_k_qubit_encoder(k)
        graph_dec, U_N, _, sv = build_graph_decoder(N, graph)

        d = 2**k
        errors = []
        fidelities = []
        enc_all_ok = True

        for trial in range(10):
            rho_in = random_density_matrix(d)
            rho_rec, enc_ok, _, _ = run_k_qubit_protocol(k, rho_in, graph_dec, U_enc_k, graph)
            err = np.linalg.norm(rho_rec - rho_in)
            fid = np.real(fidelity(rho_rec, rho_in))
            errors.append(err)
            fidelities.append(fid)
            if not enc_ok:
                enc_all_ok = False

        perfect = max(errors) < 1e-10

        print(f"    Encryption perfect:  {'✅' if enc_all_ok else '❌'}")
        print(f"    Max recovery error:  {max(errors):.2e} {'✅' if perfect else '❌'}")
        print(f"    Min fidelity:        {min(fidelities):.10f}")
        print(f"    → {'HYPOTHESIS A CONFIRMED' if perfect else 'HYPOTHESIS A FAILED'}")

HYPOTHESIS A: 2k-qubit graph → k-qubit cloning (1 clone)

────────────────────────────────────────────────────────────
  Testing Graph: Cycle Graph C_4 (k=2)
────────────────────────────────────────────────────────────
    Encryption perfect:  ❌
    Max recovery error:  1.85e-15 ✅
    Min fidelity:        1.0000000073
    → HYPOTHESIS A CONFIRMED

────────────────────────────────────────────────────────────
  Testing Graph: Random Max-Entangled Bipartite (k=2)
────────────────────────────────────────────────────────────
    Encryption perfect:  ❌
    Max recovery error:  1.92e-15 ✅
    Min fidelity:        1.0000000065
    → HYPOTHESIS A CONFIRMED

────────────────────────────────────────────────────────────
  Testing Graph: Generalized Network (k=3)
────────────────────────────────────────────────────────────
    Encryption perfect:  ❌
    Max recovery error:  2.34e-15 ✅
    Min fidelity:        1.0000000246
    → HYPOTHESIS A CONFIRMED


In [6]:
def run_multiclone_protocol(n_clones, rho_A_in, graph):
    N = 2 * n_clones
    total_q = N + 1
    dims = [2] * total_q

    psi_graph = build_graph_state(N, graph['edges'])
    perm = graph['signal_qubits'] + graph['noise_qubits']
    P_graph = build_permutation_matrix(N, perm)
    psi_reordered = P_graph @ psi_graph
    rho_resource = np.outer(psi_reordered, psi_reordered.conj())

    state = kron(rho_A_in, rho_resource)

    U_enc_n, U_dec_bell_n = build_single_qubit_codec(n_clones)

    U_enc_full = kron(U_enc_n, eye(2**n_clones))
    state_enc = U_enc_full @ state @ U_enc_full.conj().T

    U_N_graph, _ = extract_UN(psi_reordered, n_clones)

    dec_qubits = [1] + list(range(n_clones + 1, total_q))
    other_qubits = [q for q in range(total_q) if q not in dec_qubits]
    perm_dec = dec_qubits + other_qubits

    P_dec = build_permutation_matrix(total_q, perm_dec)

    U_N_on_noise = kron(eye(2), U_N_graph)
    U_dec_graph = U_dec_bell_n @ U_N_on_noise

    n_other = len(other_qubits)
    U_dec_embed = kron(U_dec_graph, eye(2**n_other))

    state_perm = P_dec @ state_enc @ P_dec.conj().T
    state_decoded = U_dec_embed @ state_perm @ U_dec_embed.conj().T
    state_final = P_dec.conj().T @ state_decoded @ P_dec

    rho_recovered = partial_trace(state_final, dims, keep=[1])
    rho_A_enc = partial_trace(state_enc, dims, keep=[0])
    enc_ok = np.allclose(rho_A_enc, I2/2, atol=1e-8)

    return rho_recovered, enc_ok, state_enc, state_final

print("=" * 70)
print("HYPOTHESIS B: 2n-qubit graph → n encrypted clones of 1 qubit")
print("=" * 70)

for N, graphs in test_graphs.items():
    n_cl = N // 2
    for graph in graphs:
        print(f"\n{'─'*60}")
        print(f"  Testing Graph: {graph['name']} (n_clones={n_cl})")
        print(f"{'─'*60}")

        errors = []
        fidelities = []
        enc_all_ok = True

        for trial in range(10):
            rho_A = random_density_matrix(2)
            rho_rec, enc_ok, _, _ = run_multiclone_protocol(n_cl, rho_A, graph)
            err = np.linalg.norm(rho_rec - rho_A)
            fid = np.real(fidelity(rho_rec, rho_A))
            errors.append(err)
            fidelities.append(fid)
            if not enc_ok:
                enc_all_ok = False

        perfect = max(errors) < 1e-10

        print(f"    Encryption perfect:  {'✅' if enc_all_ok else '❌'}")
        print(f"    Max recovery error:  {max(errors):.2e} {'✅' if perfect else '❌'}")
        print(f"    Min fidelity:        {min(fidelities):.10f}")
        print(f"    → {'HYPOTHESIS B CONFIRMED' if perfect else 'HYPOTHESIS B FAILED'}")

HYPOTHESIS B: 2n-qubit graph → n encrypted clones of 1 qubit

────────────────────────────────────────────────────────────
  Testing Graph: Cycle Graph C_4 (n_clones=2)
────────────────────────────────────────────────────────────
    Encryption perfect:  ✅
    Max recovery error:  1.46e-15 ✅
    Min fidelity:        1.0000000000
    → HYPOTHESIS B CONFIRMED

────────────────────────────────────────────────────────────
  Testing Graph: Random Max-Entangled Bipartite (n_clones=2)
────────────────────────────────────────────────────────────
    Encryption perfect:  ✅
    Max recovery error:  1.58e-15 ✅
    Min fidelity:        1.0000000000
    → HYPOTHESIS B CONFIRMED

────────────────────────────────────────────────────────────
  Testing Graph: Generalized Network (n_clones=3)
────────────────────────────────────────────────────────────
    Encryption perfect:  ✅
    Max recovery error:  1.51e-15 ✅
    Min fidelity:        1.0000000000
    → HYPOTHESIS B CONFIRMED
